In [1]:
!python --version

Python 3.11.14


## GDC

https://docs.gdc.cancer.gov/API/Users_Guide/Downloading_Files/

sample  
 ├── primary_site       (organ)  
 ├── disease_type       (broad class)  
 ├── subtype_global     (cross-cancer)  
 ├── subtype_tissue     (within-cancer)  
 └── histology          (biological family)  

### TCGA / GDC Portal

https://portal.gdc.cancer.gov/

### 🧠 Big picture (this is powerful)

- You’ve essentially built the backbone of:
- cBioPortal-like explorer
- cohort stratification engine
- patient-specific pipeline (your perturbation_agent 🔥)

### 💡 Flow

> project → project_id  
> primary_sites → pid and disease_type  
> subtypes → subtype_id  
> stages →  stage_id  
> case_id → case_ids or UUIDs  
> samples → sample_id [tumor, normal]  
> aliquots → aliquot_id  
> files → file_id  


In [2]:
import os, sys
import pandas as pd

from pathlib import Path

ROOT = Path().resolve().parent.parent
SRC = os.path.join(ROOT, "src")

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("ROOT:", ROOT)
print("SRC added:", SRC)

from libs.calc_degs_lib import CALC_DEGS
from libs.tcga_gdc_lib import *
from libs.Basic import *


ROOT: /home/flavio/uv
SRC added: /home/flavio/uv/src


### 👉 a TCGA cohort builder + data retrieval engine

Which means you can easily extend to:


> pan-cancer analyses  
> patient-specific pipelines (your perturb_agent 🔥)  
> automated workflows (Nextflow-ready)  

#### 🚀 Next steps

> Production pipeline  
  - CLI tool
  - cached queries
  - reproducible outputs

> Expression matrix builder  
  - auto-download
  - merge TSVs
  - DESeq2-ready matrix

> Patient-level analysis
  - per-case DEG
  - pathway perturbation scoring

### Defaults

In [3]:
ROOT = Path().resolve().parent
root_data = os.path.join(ROOT, "data/tcga")

gdc = GDC(root_data=root_data)

os.listdir(root_data)


['subtype_for_TCGA-KIRC.tsv',
 'degs.txt',
 'cases_for_TCGA-LUSC.tsv',
 'subtype_for_TCGA-HNSC.tsv',
 'cases_for_TCGA-PCPG.tsv',
 'Gene_Expression_Quantification_for_TCGA-BRCA_case_0045349c-69d9-4306-a403-c9c1fa836644_sample_type_Primary_Tumor_stage_unknown_fileid_0e0df72c-33c0-4e4f-939c-a4d45a6e1ea3.tsv',
 'cases_for_TCGA-THYM.tsv',
 'cases_for_TCGA-PAAD.tsv',
 'cases_for_TCGA-LAML.tsv',
 'Gene_Expression_Quantification_for_TCGA-BRCA_case_011b9b2d-ebe5-42bf-9662-d922faccc7a1_sample_type_Primary_Tumor_stage_Stage_IA_fileid_89409cf3-e710-47fd-af3c-6f8f0dec7c90.tsv',
 'cases_for_TCGA-LIHC.tsv',
 'subtype_for_TCGA-KIRP.tsv',
 'subtype_for_TCGA-CHOL.tsv',
 "samples_for_TCGA-BRCA_subtype_'other'_tumor_'other'_subtype_tissue_'other'.tsv",
 'cases_for_TCGA-MESO.tsv',
 'cases_for_TCGA-BLCA.tsv',
 'Gene_Expression_Quantification_for_TCGA-BRCA_case_157ec9e0-ca6e-4da1-9233-f976cf0db433_sample_type_Primary_Tumor_stage_Stage_IV_fileid_a127fb1e-aa3b-4b6d-98b5-c141ffba9ae7.tsv',
 'cases_for_TCGA-KIRP

### Methods

#### Get GDC programs - get_gdc_progams()

> end_point: url_gdc_project = "https://api.gdc.cancer.gov/projects"  
> "facets": "program.name"  

#### Get valid primary sites - get_primary_sites()

> end_point: url_gdc_project = "https://api.gdc.cancer.gov/projects"  
> "fields": "project_id,name,primary_site,disease_type"  

#### Get valid subtypes - get_subtypes()

> end_point: url_gdc_cases = "https://api.gdc.cancer.gov/cases"  
> facets = "diagnoses.primary_diagnosis"  

#### For each subtype → get stages  - get_stages()

> end_point: url_gdc_cases = "https://api.gdc.cancer.gov/cases"  
> "field": "diagnoses.ajcc_pathologic_stage" - AJCC diagnoses   

#### For each (subtype, stage) → get_samples()

> end_point: url_gdc_cases = "https://api.gdc.cancer.gov/cases"  
> given pid, subtype, stage  
> from cases   
> "samples.sample_id", "samples.submitter_id", "samples.sample_type"  

#### get RNA-seq files - get_expression_files_given_samples()

> given: "field": "cases.case_id", "value": case_ids  
> end_point: url_gdc_files = "https://api.gdc.cancer.gov/files"  



### Get all programs

In [4]:
force=False
verbose=True

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)


File read at '/home/flavio/uv/perturb_agent/data/tcga/programs.txt'


In [5]:
np.array(prog_list)

array(['TCGA', 'MATCH', 'TARGET', 'CGCI', 'CMI', 'APOLLO', 'BEATAML1.0',
       'CPTAC', 'MP2PRT', 'ALCHEMIST', 'CCDI', 'CCG', 'CDDP_EAGLE',
       'CTSP', 'EXCEPTIONAL_RESPONDERS', 'FM', 'HCMI', 'MMRF', 'NCICCR',
       'OHSU', 'ORGANOID', 'RC', 'REBC', 'TRIO', 'VAREPOP', 'WCDT'],
      dtype='<U22')

### Primary sites given a program

In [6]:
gdc.url_gdc_project

'https://api.gdc.cancer.gov/projects'

In [7]:
force=False
verbose=True

program = 'TCGA'

df_psi = gdc.get_primary_sites(program=program, force=force, verbose=verbose)

print(len(df_psi))
df_psi.head(12)


Table opened ((33, 5)) at '/home/flavio/uv/perturb_agent/data/tcga/primary_site_program_TCGA.tsv'
33


,pid,primary_site,project_id,disease_type,name
0,TCGA-ACC,Adrenal gland,TCGA-ACC,Adenomas and Adenocarcinomas,Adrenocortical Carcinoma
1,TCGA-PCPG,"Adrenal gland, Retroperitoneum and peritoneum,...",TCGA-PCPG,Paragangliomas and Glomus Tumors,Pheochromocytoma and Paraganglioma
2,TCGA-BLCA,Bladder,TCGA-BLCA,"Epithelial Neoplasms, NOS, Squamous Cell Neopl...",Bladder Urothelial Carcinoma
3,TCGA-LGG,Brain,TCGA-LGG,Gliomas,Brain Lower Grade Glioma
4,TCGA-GBM,Brain,TCGA-GBM,"Not Reported, Gliomas",Glioblastoma Multiforme
5,TCGA-BRCA,Breast,TCGA-BRCA,"Adnexal and Skin Appendage Neoplasms, Adenomas...",Breast Invasive Carcinoma
6,TCGA-LUAD,Bronchus and lung,TCGA-LUAD,"Ductal and Lobular Neoplasms, Cystic, Mucinous...",Lung Adenocarcinoma
7,TCGA-LUSC,Bronchus and lung,TCGA-LUSC,"Squamous Cell Neoplasms, Adenomas and Adenocar...",Lung Squamous Cell Carcinoma
8,TCGA-MESO,"Bronchus and lung, Heart, mediastinum, and pleura",TCGA-MESO,Mesothelial Neoplasms,Mesothelioma
9,TCGA-CESC,"Cervix uteri, Ovary",TCGA-CESC,"Squamous Cell Neoplasms, Cystic, Mucinous and ...",Cervical Squamous Cell Carcinoma and Endocervi...


In [8]:
df_psi.tail(3)

,pid,primary_site,project_id,disease_type,name
30,TCGA-THCA,Thyroid gland,TCGA-THCA,"Squamous Cell Neoplasms, Epithelial Neoplasms,...",Thyroid Carcinoma
31,TCGA-UCS,"Uterus, NOS, Corpus uteri",TCGA-UCS,"Basal Cell Neoplasms, Complex Mixed and Stroma...",Uterine Carcinosarcoma
32,TCGA-UCEC,"Uterus, NOS, Corpus uteri",TCGA-UCEC,"Not Reported, Cystic, Mucinous and Serous Neop...",Uterine Corpus Endometrial Carcinoma


### Subtypes explanation

👉 GDC does NOT give you clean biological subtypes
👉 You must derive them yourself

This is actually a feature, not a bug — because:

> you control granularity  
> you can harmonize across cancers  
> you avoid noisy labels  


💡 If you want to level this up

I can help you build:

🔬 A cross-TCGA subtype harmonizer
maps all projects → unified subtype ontology
handles synonyms (adenocarcinoma, NOS, etc.)
outputs clean ML-ready labels

👉 That would massively improve your perturbation analysis.

In [9]:
i = 3
pid = df_psi.iloc[i].pid
primary_site = df_psi.iloc[i].primary_site

print(i, pid, primary_site)

3 TCGA-LGG Brain


- Brain tumors do NOT use AJCC staging (3 - TCGA-LGG Brain)

In [10]:
force=False
verbose=True

pid = df_psi.iloc[i].pid
primary_site = df_psi.iloc[i].primary_site

print(i, pid, primary_site)

df_cases, df_subt, df_prof = gdc.get_cases_and_subtypes(pid=pid, batch_size=200, do_filter=False, force=force, verbose=verbose)

print("df_cases", len(df_cases), "df_subt", len(df_subt))

3 TCGA-LGG Brain
Table opened ((516, 24)) at '/home/flavio/uv/perturb_agent/data/tcga/cases_for_TCGA-LGG.tsv'
df_cases 516 df_subt 2


### df_subt

In [11]:
df_subt

,pid,subtype_global,tumor_class,subtype_tissue,stage,n
0,TCGA-LGG,other,glioma,other,unknown,466
1,TCGA-LGG,other,other,other,unknown,50


### df_cases

In [12]:
df_cases.head(3)

,primary_site,disease_type,case_id,diagnoses,pid,subtype_global,stage_ajcc,primary_diagnosis,tumor_grade,stage_clin,...,disease_type_norm,diagnosis_norm,tumor_class,histology,subtype_tissue,consistency,validity,n,frac,sstage
0,Brain,Gliomas,2cf49bac-dea1-45ff-9e1d-09f5a2ac5215,"[{'primary_diagnosis': 'Oligodendroglioma, NOS...",TCGA-LGG,other,unknown,"Oligodendroglioma, NOS",G3,NaN,...,gliomas,oligodendroglioma,glioma,other,other,ok,ambiguous,1,0.001938,missing
1,Brain,Gliomas,183dd089-e932-4be2-b252-0e8572e7da4e,"[{'primary_diagnosis': 'Mixed glioma', 'tumor_...",TCGA-LGG,other,unknown,Mixed glioma,G2,NaN,...,gliomas,mixed glioma,glioma,other,other,ok,ambiguous,1,0.001938,missing
2,Brain,Gliomas,1f33aebc-edc1-42c5-baa9-e4b0a06fcb26,"[{'primary_diagnosis': 'Astrocytoma, NOS', 'tu...",TCGA-LGG,other,unknown,"Astrocytoma, NOS",G2,NaN,...,gliomas,astrocytoma,glioma,other,other,ok,ambiguous,1,0.001938,missing


In [13]:
cols = ["pid", "case_id", "subtype_global", "tumor_class", "subtype_tissue", "stage"]

df_cases[cols].head(12)

,pid,case_id,subtype_global,tumor_class,subtype_tissue,stage
0,TCGA-LGG,2cf49bac-dea1-45ff-9e1d-09f5a2ac5215,other,glioma,other,unknown
1,TCGA-LGG,183dd089-e932-4be2-b252-0e8572e7da4e,other,glioma,other,unknown
2,TCGA-LGG,1f33aebc-edc1-42c5-baa9-e4b0a06fcb26,other,glioma,other,unknown
3,TCGA-LGG,cb598780-9e42-4167-b487-eec90ad4f36f,other,glioma,other,unknown
4,TCGA-LGG,dc7ec60b-5ea5-48a4-8908-55caf9214272,other,glioma,other,unknown
5,TCGA-LGG,60b30a95-3e4c-413a-9de6-16a5bca163c4,other,glioma,other,unknown
6,TCGA-LGG,b7cd3b44-ef6a-4207-a44f-14f5b3b56ed4,other,glioma,other,unknown
7,TCGA-LGG,a0b7c7c8-a6f2-41ba-9e61-9b7f8fbbbe1f,other,glioma,other,unknown
8,TCGA-LGG,c16d9f69-26f9-4ec7-b2d5-ff9f7dacfe0f,other,glioma,other,unknown
9,TCGA-LGG,e09f93e1-1649-48fa-9e2e-d13ac6d50b97,other,glioma,other,unknown


### Get all cases and their classifications

In [14]:

force=True
verbose=True

run_all = False

if run_all:

    for i in range(len(df_psi)):
        print(i, end=' ')
        pid = df_psi.iloc[i].pid
        primary_site = df_psi.iloc[i].primary_site

        print(pid, primary_site)

        df_cases, df_subt, df_prof = gdc.get_cases_and_subtypes(pid=pid, batch_size=200, do_filter=False, force=force, verbose=verbose)



### Search for samples and files for each case given

- input: pid, subtype_global, tumor_class, subtype_tissue, and stage

In [15]:
force=False
verbose=True

ipsi=5
pid = df_psi.iloc[ipsi].pid
primary_site = df_psi.iloc[ipsi].primary_site

print(i, pid, primary_site)

df_cases, df_subt, df_prof = gdc.get_cases_and_subtypes(pid=pid, batch_size=200, do_filter=False, force=force, verbose=verbose)
df_subt

3 TCGA-BRCA Breast
Table opened ((1098, 24)) at '/home/flavio/uv/perturb_agent/data/tcga/cases_for_TCGA-BRCA.tsv'


,pid,subtype_global,tumor_class,subtype_tissue,stage,n
0,TCGA-BRCA,other,other,other,Stage IIA,263
1,TCGA-BRCA,lobular,other,breast_lobular,unknown,223
2,TCGA-BRCA,other,other,other,Stage IIB,180
3,TCGA-BRCA,other,other,other,Stage IIIA,107
4,TCGA-BRCA,other,other,other,unknown,76
5,TCGA-BRCA,other,other,other,Stage IA,69
6,TCGA-BRCA,other,other,other,Stage I,67
7,TCGA-BRCA,other,other,other,Stage IIIC,29
8,TCGA-BRCA,other,other,other,Stage IIIB,18
9,TCGA-BRCA,adenocarcinoma_generic,adenocarcinoma,adenocarcinoma_generic,unknown,18


In [16]:
cols

['pid', 'case_id', 'subtype_global', 'tumor_class', 'subtype_tissue', 'stage']

In [17]:
isubt = 3
df_subt.iloc[isubt]

pid                TCGA-BRCA
subtype_global         other
tumor_class            other
subtype_tissue         other
stage             Stage IIIA
n                        107
Name: 3, dtype: object

In [18]:
force=False
verbose=True

row = df_subt.iloc[isubt]

pid = row.pid
subtype_global = row.subtype_global
tumor_class = row.tumor_class
subtype_tissue = row.subtype_tissue
stage = None # row.stage

# s_case = f"{pid} subtype '{subtype_global}' tumor '{tumor_class}' subtype_tissue '{subtype_tissue}' stage '{stage}'"
# print(s_case)
df_samples = gdc.get_samples_for_pid_subtypes(pid=pid, subtype_global=subtype_global,
                                             tumor_class=tumor_class, subtype_tissue=subtype_tissue,
                                             batch_size=200, force=force, verbose=verbose)

print(len(df_samples))

df_samples.head(3)


Table opened ((1098, 24)) at '/home/flavio/uv/perturb_agent/data/tcga/cases_for_TCGA-BRCA.tsv'
>>> TCGA-BRCA subtype 'other' tumor 'other' subtype_tissue 'other'
Table opened ((587, 11)) at '/home/flavio/uv/perturb_agent/data/tcga/samples_for_TCGA-BRCA_subtype_'other'_tumor_'other'_subtype_tissue_'other'.tsv'
587


,case_id,sample_id,sample_type,file_id,file_name,data_type,pid,subtype_global,tumor_class,subtype_tissue,stage
0,01263518-5f7c-49dc-8d7e-84b0c03a6a63,5d1a5545-6934-41e5-b41b-8e0ee88e6ec6,Primary Tumor,7ccc5a5b-d3f1-46da-8540-d8fbb7c6c06e,TCGA-A8-A07W-01A-01-TSA.eae41004-b3f5-4528-9e9...,Slide Image,TCGA-BRCA,other,other,other,Stage IV
1,01263518-5f7c-49dc-8d7e-84b0c03a6a63,5d1a5545-6934-41e5-b41b-8e0ee88e6ec6,Primary Tumor,8f355183-b72c-407e-8f25-f5eec7f115cc,e06580e9-0b4a-48f7-9522-d54229aefb2d.rna_seq.t...,Aligned Reads,TCGA-BRCA,other,other,other,Stage IV
2,01263518-5f7c-49dc-8d7e-84b0c03a6a63,5d1a5545-6934-41e5-b41b-8e0ee88e6ec6,Primary Tumor,e209288d-df03-4a61-a84f-eec243a2c43f,TCGA-BRCA.3e06faf7-1e08-4c15-9aee-8891de8cad5c...,Gene Level Copy Number,TCGA-BRCA,other,other,other,Stage IV


In [19]:
Counter(df_samples.stage)

Counter({'Stage IA': 289, 'Stage IIA': 105, 'unknown': 102, 'Stage IV': 91})

In [20]:
df_samples.sample_type.unique()

array(['Primary Tumor', 'Blood Derived Normal', 'Solid Tissue Normal'],
      dtype=object)

In [21]:
df_prof

,primary_diagnosis,stage,tumor_grade,diagnosis_conflict,n_diagnoses
0,None,None,None,False,0
1,None,None,None,False,0
2,None,None,None,False,0
3,None,None,None,False,0
4,None,None,None,False,0
...,...,...,...,...,...
1093,None,None,None,False,0
1094,None,None,None,False,0
1095,None,None,None,False,0
1096,None,None,None,False,0


In [22]:
df_cases.stage.unique()

array(['unknown', 'Stage IIIA', 'Stage IA', 'Stage IIB', 'Stage IIA',
       'Stage IIIB', 'Stage IIIC', 'Stage I', 'Stage IV', 'Stage X',
       'Stage II', 'Stage IB'], dtype=object)

In [23]:
np.unique(df_samples.data_type)

array(['Aggregated Somatic Mutation', 'Aligned Reads',
       'Allele-specific Copy Number Segment',
       'Annotated Somatic Mutation', 'Copy Number Segment',
       'Gene Expression Quantification', 'Gene Level Copy Number',
       'Intermediate Analysis Archive',
       'Isoform Expression Quantification', 'Masked Copy Number Segment',
       'Masked Intensities', 'Masked Somatic Mutation',
       'Methylation Beta Value', 'Pathology Report',
       'Protein Expression Quantification', 'Raw Intensities',
       'Raw Simple Somatic Mutation', 'Simple Germline Variation',
       'Slide Image', 'Splice Junction Quantification',
       'Structural Rearrangement', 'Transcript Fusion',
       'miRNA Expression Quantification'], dtype=object)

In [24]:
data_type = 'Gene Expression Quantification'
df_samples2 = df_samples[df_samples.data_type == data_type]
print(len(df_samples2))
df_samples2

7


,case_id,sample_id,sample_type,file_id,file_name,data_type,pid,subtype_global,tumor_class,subtype_tissue,stage
3,01263518-5f7c-49dc-8d7e-84b0c03a6a63,5d1a5545-6934-41e5-b41b-8e0ee88e6ec6,Primary Tumor,aa89d193-05a1-4f7f-9291-b9cf646656de,e06580e9-0b4a-48f7-9522-d54229aefb2d.rna_seq.a...,Gene Expression Quantification,TCGA-BRCA,other,other,other,Stage IV
103,011b9b2d-ebe5-42bf-9662-d922faccc7a1,d6023f30-0d07-4839-98d1-0b22b366cc51,Primary Tumor,89409cf3-e710-47fd-af3c-6f8f0dec7c90,e6bad6ec-c178-4684-99e8-2504781a022b.rna_seq.a...,Gene Expression Quantification,TCGA-BRCA,other,other,other,Stage IA
108,011b9b2d-ebe5-42bf-9662-d922faccc7a1,e716eb8a-dccd-4f17-9bdc-e64dc2f0fd69,Primary Tumor,a5c799bc-1332-4127-8fda-84da04084d25,de01516e-43f0-4f96-8ac6-ab543a314829.rna_seq.a...,Gene Expression Quantification,TCGA-BRCA,other,other,other,Stage IA
216,011b9b2d-ebe5-42bf-9662-d922faccc7a1,d6023f30-0d07-4839-98d1-0b22b366cc51,Primary Tumor,2b519fff-dd2a-458e-ac2b-573167aeb7f2,a02fb212-a6cc-40b2-9d31-481fc1ce0911.rna_seq.a...,Gene Expression Quantification,TCGA-BRCA,other,other,other,Stage IA
365,00a2d166-78c9-4687-a195-3d6315c27574,2e4b5ba3-ccc8-40f6-80ac-edaea1685204,Primary Tumor,e14858ca-8bb5-4d6d-906c-5d62722d90f8,fd6f4c9b-ee43-4939-8cfa-2e447aedbcf3.rna_seq.a...,Gene Expression Quantification,TCGA-BRCA,other,other,other,Stage IIA
424,0045349c-69d9-4306-a403-c9c1fa836644,f66358d4-9319-4c23-983d-ab5056daacf1,Primary Tumor,0e0df72c-33c0-4e4f-939c-a4d45a6e1ea3,36125e17-48fd-4eea-874c-ed2e2e218402.rna_seq.a...,Gene Expression Quantification,TCGA-BRCA,other,other,other,unknown
560,001cef41-ff86-4d3f-a140-a647ac4b10a1,92058c44-a484-4e08-b1fe-dfe2f03a0aa1,Primary Tumor,41e79241-b5a4-4541-848b-e20e693e8ee3,22c2b380-799e-4fad-ae38-46a916c592d5.rna_seq.a...,Gene Expression Quantification,TCGA-BRCA,other,other,other,Stage IA


In [25]:
df_samples2.groupby(['case_id', 'sample_type', 'data_type']).size()

case_id                               sample_type    data_type                     
001cef41-ff86-4d3f-a140-a647ac4b10a1  Primary Tumor  Gene Expression Quantification    1
0045349c-69d9-4306-a403-c9c1fa836644  Primary Tumor  Gene Expression Quantification    1
00a2d166-78c9-4687-a195-3d6315c27574  Primary Tumor  Gene Expression Quantification    1
011b9b2d-ebe5-42bf-9662-d922faccc7a1  Primary Tumor  Gene Expression Quantification    3
01263518-5f7c-49dc-8d7e-84b0c03a6a63  Primary Tumor  Gene Expression Quantification    1
dtype: int64

In [26]:
pid

'TCGA-BRCA'

In [27]:
df_cases, _, _ = gdc.get_cases_and_subtypes(pid=pid, batch_size=200, do_filter=False, 
											         force=False, verbose=True)

cols = ['subtype_global', 'tumor_class', 'subtype_tissue', 'stage']
df_cases[cols]

Table opened ((1098, 24)) at '/home/flavio/uv/perturb_agent/data/tcga/cases_for_TCGA-BRCA.tsv'


,subtype_global,tumor_class,subtype_tissue,stage
0,other,other,other,unknown
1,other,other,other,Stage IIIA
2,other,other,other,Stage IA
3,other,other,other,Stage IIB
4,other,other,other,Stage IA
...,...,...,...,...
1093,other,other,other,Stage IIA
1094,other,other,other,Stage IIA
1095,other,other,other,Stage IIB
1096,lobular,other,breast_lobular,unknown


In [28]:
gdc.df_cases.head(2)

,primary_site,disease_type,case_id,diagnoses,pid,subtype_global,stage_ajcc,primary_diagnosis,tumor_grade,stage_clin,...,disease_type_norm,diagnosis_norm,tumor_class,histology,subtype_tissue,consistency,validity,n,frac,sstage
0,Breast,Complex Epithelial Neoplasms,e3935ce4-64d3-4a66-ba11-d308b844b410,"[{'primary_diagnosis': 'Metaplastic carcinoma,...",TCGA-BRCA,other,unknown,"Metaplastic carcinoma, NOS",NaN,NaN,...,complex epithelial neoplasms,metaplastic carcinoma,other,other,other,ok,ambiguous,1,0.000911,missing
1,Breast,Ductal and Lobular Neoplasms,e3b555aa-7f0a-49c6-9b13-182c61a144c1,[{'primary_diagnosis': 'Infiltrating duct carc...,TCGA-BRCA,other,Stage IIIA,"Infiltrating duct carcinoma, NOS",NaN,NaN,...,ductal and lobular neoplasms,infiltrating duct carcinoma,other,other,other,ok,ambiguous,1,0.000911,III


In [29]:
dfu = gdc.group_file_types(df_samples)
dfu

,data_type,n
0,Annotated Somatic Mutation,132
1,Raw Simple Somatic Mutation,92
2,Aligned Reads,49
3,Structural Rearrangement,44
4,Gene Level Copy Number,35
5,Allele-specific Copy Number Segment,30
6,Transcript Fusion,28
7,Slide Image,23
8,Copy Number Segment,21
9,Masked Somatic Mutation,14


In [30]:
lista = list(dfu.data_type)
lista.sort()

"; ".join(lista)

'Aggregated Somatic Mutation; Aligned Reads; Allele-specific Copy Number Segment; Annotated Somatic Mutation; Copy Number Segment; Gene Expression Quantification; Gene Level Copy Number; Intermediate Analysis Archive; Isoform Expression Quantification; Masked Copy Number Segment; Masked Intensities; Masked Somatic Mutation; Methylation Beta Value; Pathology Report; Protein Expression Quantification; Raw Intensities; Raw Simple Somatic Mutation; Simple Germline Variation; Slide Image; Splice Junction Quantification; Structural Rearrangement; Transcript Fusion; miRNA Expression Quantification'

### Get Any table (for tumor and control)

In [31]:
np.unique(df_samples.data_type)

array(['Aggregated Somatic Mutation', 'Aligned Reads',
       'Allele-specific Copy Number Segment',
       'Annotated Somatic Mutation', 'Copy Number Segment',
       'Gene Expression Quantification', 'Gene Level Copy Number',
       'Intermediate Analysis Archive',
       'Isoform Expression Quantification', 'Masked Copy Number Segment',
       'Masked Intensities', 'Masked Somatic Mutation',
       'Methylation Beta Value', 'Pathology Report',
       'Protein Expression Quantification', 'Raw Intensities',
       'Raw Simple Somatic Mutation', 'Simple Germline Variation',
       'Slide Image', 'Splice Junction Quantification',
       'Structural Rearrangement', 'Transcript Fusion',
       'miRNA Expression Quantification'], dtype=object)

In [32]:
df_samples2 = df_samples[df_samples.data_type == 'Gene Expression Quantification'].copy().reset_index(drop=True)
df_samples2.head(5)

,case_id,sample_id,sample_type,file_id,file_name,data_type,pid,subtype_global,tumor_class,subtype_tissue,stage
0,01263518-5f7c-49dc-8d7e-84b0c03a6a63,5d1a5545-6934-41e5-b41b-8e0ee88e6ec6,Primary Tumor,aa89d193-05a1-4f7f-9291-b9cf646656de,e06580e9-0b4a-48f7-9522-d54229aefb2d.rna_seq.a...,Gene Expression Quantification,TCGA-BRCA,other,other,other,Stage IV
1,011b9b2d-ebe5-42bf-9662-d922faccc7a1,d6023f30-0d07-4839-98d1-0b22b366cc51,Primary Tumor,89409cf3-e710-47fd-af3c-6f8f0dec7c90,e6bad6ec-c178-4684-99e8-2504781a022b.rna_seq.a...,Gene Expression Quantification,TCGA-BRCA,other,other,other,Stage IA
2,011b9b2d-ebe5-42bf-9662-d922faccc7a1,e716eb8a-dccd-4f17-9bdc-e64dc2f0fd69,Primary Tumor,a5c799bc-1332-4127-8fda-84da04084d25,de01516e-43f0-4f96-8ac6-ab543a314829.rna_seq.a...,Gene Expression Quantification,TCGA-BRCA,other,other,other,Stage IA
3,011b9b2d-ebe5-42bf-9662-d922faccc7a1,d6023f30-0d07-4839-98d1-0b22b366cc51,Primary Tumor,2b519fff-dd2a-458e-ac2b-573167aeb7f2,a02fb212-a6cc-40b2-9d31-481fc1ce0911.rna_seq.a...,Gene Expression Quantification,TCGA-BRCA,other,other,other,Stage IA
4,00a2d166-78c9-4687-a195-3d6315c27574,2e4b5ba3-ccc8-40f6-80ac-edaea1685204,Primary Tumor,e14858ca-8bb5-4d6d-906c-5d62722d90f8,fd6f4c9b-ee43-4939-8cfa-2e447aedbcf3.rna_seq.a...,Gene Expression Quantification,TCGA-BRCA,other,other,other,Stage IIA


In [33]:
force=False
verbose=True

isamp=0
row = df_samples2.iloc[isamp]

pid = row.pid
case_id = row.case_id
file_type = row.data_type
file_id = row.file_id
sample_type = row.sample_type
stage = row.stage


print(f"{pid}, {case_id}, {file_type}, {file_id}")

dft = gdc.get_table_given_fileID(pid=pid, case_id=case_id, 
                                 file_type=file_type, sample_type=sample_type, stage=stage, file_id=file_id, 
                                 force=force, verbose=verbose)
print(len(dft))
dft.head(6)

TCGA-BRCA, 01263518-5f7c-49dc-8d7e-84b0c03a6a63, Gene Expression Quantification, aa89d193-05a1-4f7f-9291-b9cf646656de
Table opened ((60660, 9)) at '/home/flavio/uv/perturb_agent/data/tcga/Gene_Expression_Quantification_for_TCGA-BRCA_case_01263518-5f7c-49dc-8d7e-84b0c03a6a63_sample_type_Primary_Tumor_stage_Stage_IV_fileid_aa89d193-05a1-4f7f-9291-b9cf646656de.tsv'
60660


,gene_id,symbol,gene_type,unstranded,counts,stranded_second,tpm_unstranded,fpkm_unstranded,fpkm_uq_unstranded
0,ENSG00000000003,TSPAN6,protein_coding,911,446,465,12.8633,3.2599,3.1661
1,ENSG00000000005,TNMD,protein_coding,22,12,10,0.9547,0.2419,0.2350
2,ENSG00000000419,DPM1,protein_coding,4349,2189,2160,230.7761,58.4850,56.8023
3,ENSG00000000457,SCYL3,protein_coding,1420,1109,1155,13.2135,3.3487,3.2523
4,ENSG00000000460,C1orf112,protein_coding,1197,1075,1036,12.8419,3.2545,3.1608
5,ENSG00000000938,FGR,protein_coding,679,366,313,12.8589,3.2588,3.1650


### Get Expression table (for tumor and control)

#### 🧪 Mental model (important)

| Step	|  Needs strand? | Output | Column |
|---------|---------|---------|---------|
| Read assignment	|  YES |  counts | stranded_first |
| Normalization	| NO | TPM / FPKM | both: unstranded |
| Biological meaning | NO | gene expression| both: unstranded |

In [34]:
np.unique(df_samples2.stage)

array(['Stage IA', 'Stage IIA', 'Stage IV', 'unknown'], dtype=object)

In [35]:
len(df_samples2)

7

In [36]:
file_type

'Gene Expression Quantification'

In [37]:
force=False
verbose=False

dft = pd.DataFrame()

for i, row in df_samples2.iterrows():
    pid = row.pid
    case_id = row.case_id
    file_type = row.data_type
    file_id = row.file_id
    sample_type = row.sample_type
    stage = row.stage

    print(f"{i}) {pid}, {case_id}, {file_type}, {file_id}")

    dft = gdc.get_table_given_fileID(pid=pid, case_id=case_id, 
                                     sample_type=sample_type, stage=stage, 
                                     file_type=file_type, file_id=file_id, 
                                     force=force, verbose=verbose)


dft.head(6)



0) TCGA-BRCA, 01263518-5f7c-49dc-8d7e-84b0c03a6a63, Gene Expression Quantification, aa89d193-05a1-4f7f-9291-b9cf646656de
1) TCGA-BRCA, 011b9b2d-ebe5-42bf-9662-d922faccc7a1, Gene Expression Quantification, 89409cf3-e710-47fd-af3c-6f8f0dec7c90
2) TCGA-BRCA, 011b9b2d-ebe5-42bf-9662-d922faccc7a1, Gene Expression Quantification, a5c799bc-1332-4127-8fda-84da04084d25
3) TCGA-BRCA, 011b9b2d-ebe5-42bf-9662-d922faccc7a1, Gene Expression Quantification, 2b519fff-dd2a-458e-ac2b-573167aeb7f2
4) TCGA-BRCA, 00a2d166-78c9-4687-a195-3d6315c27574, Gene Expression Quantification, e14858ca-8bb5-4d6d-906c-5d62722d90f8
5) TCGA-BRCA, 0045349c-69d9-4306-a403-c9c1fa836644, Gene Expression Quantification, 0e0df72c-33c0-4e4f-939c-a4d45a6e1ea3
6) TCGA-BRCA, 001cef41-ff86-4d3f-a140-a647ac4b10a1, Gene Expression Quantification, 41e79241-b5a4-4541-848b-e20e693e8ee3


,gene_id,symbol,gene_type,unstranded,counts,stranded_second,tpm_unstranded,fpkm_unstranded,fpkm_uq_unstranded
0,ENSG00000000003,TSPAN6,protein_coding,850,427,424,15.3427,4.2169,3.7572
1,ENSG00000000005,TNMD,protein_coding,5,2,3,0.2774,0.0762,0.0679
2,ENSG00000000419,DPM1,protein_coding,1680,835,845,113.9613,31.3219,27.9074
3,ENSG00000000457,SCYL3,protein_coding,1559,1169,1184,18.5449,5.0970,4.5414
4,ENSG00000000460,C1orf112,protein_coding,402,637,597,5.5132,1.5153,1.3501
5,ENSG00000000938,FGR,protein_coding,334,177,158,8.0859,2.2224,1.9801


In [38]:
np.unique(df_samples.data_type)

array(['Aggregated Somatic Mutation', 'Aligned Reads',
       'Allele-specific Copy Number Segment',
       'Annotated Somatic Mutation', 'Copy Number Segment',
       'Gene Expression Quantification', 'Gene Level Copy Number',
       'Intermediate Analysis Archive',
       'Isoform Expression Quantification', 'Masked Copy Number Segment',
       'Masked Intensities', 'Masked Somatic Mutation',
       'Methylation Beta Value', 'Pathology Report',
       'Protein Expression Quantification', 'Raw Intensities',
       'Raw Simple Somatic Mutation', 'Simple Germline Variation',
       'Slide Image', 'Splice Junction Quantification',
       'Structural Rearrangement', 'Transcript Fusion',
       'miRNA Expression Quantification'], dtype=object)

### Get Normal and Tumor datasets given case_id and data_type

In [39]:
df_samples2.head(2)

,case_id,sample_id,sample_type,file_id,file_name,data_type,pid,subtype_global,tumor_class,subtype_tissue,stage
0,01263518-5f7c-49dc-8d7e-84b0c03a6a63,5d1a5545-6934-41e5-b41b-8e0ee88e6ec6,Primary Tumor,aa89d193-05a1-4f7f-9291-b9cf646656de,e06580e9-0b4a-48f7-9522-d54229aefb2d.rna_seq.a...,Gene Expression Quantification,TCGA-BRCA,other,other,other,Stage IV
1,011b9b2d-ebe5-42bf-9662-d922faccc7a1,d6023f30-0d07-4839-98d1-0b22b366cc51,Primary Tumor,89409cf3-e710-47fd-af3c-6f8f0dec7c90,e6bad6ec-c178-4684-99e8-2504781a022b.rna_seq.a...,Gene Expression Quantification,TCGA-BRCA,other,other,other,Stage IA


In [40]:
df_samples2.columns

Index(['case_id', 'sample_id', 'sample_type', 'file_id', 'file_name',
       'data_type', 'pid', 'subtype_global', 'tumor_class', 'subtype_tissue',
       'stage'],
      dtype='object')

In [41]:
data_type

'Gene Expression Quantification'

In [ ]:
case_id_list = list(np.unique(df_samples2.case_id))
print(len(case_id_list))


df_normal, df_tumor = gdc.get_tumor_normal_tables(df_samples=df_samples, case_id_list=case_id_list, data_type=data_type)

len(df_normal), len(df_tumor)

5
No normal samples found for case ID ['001cef41-ff86-4d3f-a140-a647ac4b10a1'
 '0045349c-69d9-4306-a403-c9c1fa836644'
 '00a2d166-78c9-4687-a195-3d6315c27574'
 '011b9b2d-ebe5-42bf-9662-d922faccc7a1'
 '01263518-5f7c-49dc-8d7e-84b0c03a6a63'] and data type Gene Expression Quantification.


(0, 0)

In [ ]:

def return_normal_tumor(x:Any, term:str) -> bool:
    if not isinstance(x,str):
        return False
    
    x = x.lower()

    if 'blood' in x:
        return False

    if term in x:
        return True

    return False

return_normal_tumor('Normal blood', 'normal')

In [ ]:
df_normal

In [ ]:
df_tumor

### Merge Expression Tables for DEGs' calculation

In [ ]:
dfa_normal, dfa_tumor = gdc.merge_normal_tumor_tables(pid=pid, df_normal=df_normal, df_tumor=df_tumor)

dfa_normal.shape, dfa_tumor.shape

In [ ]:
dfa_normal.head(3)

In [ ]:
dfa_tumor.head(3)

In [ ]:
i=100
dfa_tumor.iloc[i:i+30]

### Calc DEGs

In [ ]:
cdegs = CALC_DEGS(root_data=root_data)

cdegs.libs_dir, cdegs.root_data

In [ ]:
dfa_normal2 = cdegs.deduplicate_by_max_reads(dfa_normal)
len(dfa_normal2), len(dfa_normal)

In [ ]:
dfa_tumor2 = cdegs.deduplicate_by_max_reads(dfa_tumor)
len(dfa_tumor2), len(dfa_tumor)

In [ ]:
df_lfc = cdegs.run_deg_rscript(df_tumor=dfa_tumor2, df_normal=dfa_normal2,
                                method="edger",  manual_dispersion=0.1, min_total_count=10, 
                                merge_how="inner", keep_temp=False)
print(len(df_lfc))

In [ ]:
df_lfc = df_lfc.rename(columns={"log2FoldChange": "lfc", "padj": "fdr"})
df_lfc.head(3)

In [ ]:
lfc_cutoff = 1.0
fdr_cutoff = .05

df_degs = df_lfc[ (df_lfc.lfc >= lfc_cutoff) & (df_lfc.fdr < fdr_cutoff)].copy()
print(len(df_degs))

df_degs.head(3)

In [ ]:
fname = 'degs.txt'

text = "\n".join(df_degs.symbol)

write_txt(text, fname, root_data)

In [ ]:
TCGA BRCA breast cancer, subtypes 'other', stage IV